In [1]:
import numpy as np
import pandas as pd

In [2]:
variants = pd.read_csv("/Users/harperwood/Downloads/DMS_ProteinGym_substitutions/P53_HUMAN_Giacomelli_2018_Null_Etoposide.csv")

In [3]:
variants.head()

,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,M1A,AEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,-0.788753,0
1,M1C,CEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,-1.969077,0
2,M1Y,YEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,-1.333315,0
3,M1W,WEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,-2.219256,0
4,M1V,VEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,-0.669727,0


In [7]:
# atchley factor dictionary
atchley = {
    'A': [-0.591, -1.302, -0.733,  1.570, -0.146],
    'C': [-1.343,  0.465, -0.862, -1.020, -0.255],
    'D': [ 1.050,  0.302, -3.656, -0.259, -3.242],
    'E': [ 1.357, -1.453,  1.477,  0.113, -0.837],
    'F': [-1.006, -0.590,  1.891, -0.397,  0.412],
    'G': [-0.384,  1.652,  1.330,  1.045,  2.064],
    'H': [ 0.336, -0.417, -1.673, -1.474, -0.078],
    'I': [-1.239, -0.547,  2.131,  0.393,  0.816],
    'K': [ 1.831, -0.561,  0.533, -0.277,  1.648],
    'L': [-1.019, -0.987, -1.505,  1.266, -0.912],
    'M': [-0.663, -1.524,  2.219, -1.005,  1.212],
    'N': [ 0.945,  0.828,  1.299, -0.169,  0.933],
    'P': [ 0.189,  2.081, -1.628,  0.421, -1.392],
    'Q': [ 0.931, -0.179, -3.005, -0.503, -1.853],
    'R': [ 1.538, -0.055,  1.502,  0.440,  2.897],
    'S': [-0.228,  1.399, -4.760,  0.670, -2.647],
    'T': [-0.032,  0.326,  2.213,  0.908,  1.313],
    'V': [-1.337, -0.279, -0.544,  1.242, -1.262],
    'W': [-0.595,  0.009,  0.672, -2.128, -0.184],
    'Y': [ 0.260,  0.830,  3.097, -0.838,  1.512]
}

I'm thinking that we don't want the data to be extremely sparse, like it would have been with the orignal setup (I originally suggested encoding full sequence, subtracting to get zeroes at every position except the variant). Instead, this processes only the 'mutant string' and returns a 6-entry vector, the first 5 of which represent the differences between the wt and mutant in all 5 Atchley factors, plus a sixth encoding the position along the sequence.

In [ ]:
# data extraction from column 'mutant'
## atchley factor conversion for mutant and wildtype followed by subtraction
## position encoded as a sixth feature to avoid having extremely sparse samples
#### ex. M1A -> wt atchley for M, mut atchley for A, position = 1

def encode_mutant(mutant_str):
    wt_aa = mutant_str[0]
    mut_aa = mutant_str[-1]
    position = int(mutant_str[1:-1])
    delta = np.array(atchley[mut_aa]) - np.array(atchley[wt_aa])
    return np.append(delta, position)


# convert data and collate rows (1 row = 1 mutant)
X = np.vstack([encode_mutant(m) for m in variants['mutant']])
